[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/14_kv_cache.ipynb)

# 🔴 困难：KV 缓存注意力

实现带有 **KV 缓存**的多头注意力，用于高效的自回归生成。

在 LLM 推理过程中，每一步都重新计算所有的键/值投影会造成浪费。
**KV 缓存**存储之前计算好的 K 和 V 张量，这样只有新的 token 才需要投影。

### 函数签名
```python
class KVCacheAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor, cache=None) -> tuple[torch.Tensor, tuple]:
        # x: (B, S_new, D) — 新的 token
        # cache: None 或 (K_past, V_past)，各为 (B, num_heads, S_past, d_k)
        # 返回: (output, (K_all, V_all))
```

### 要求
- 继承自 `nn.Module`
- `self.W_q`、`self.W_k`、`self.W_v`、`self.W_o`：`nn.Linear` 投影层
- 当 `cache=None`（预填充）时：应用**因果掩码**，将所有 K/V 作为缓存返回
- 当提供 `cache`（解码）时：将新 K/V 与缓存拼接，单 token 解码无需因果掩码
- 增量解码必须与完整前向传播产生**完全相同**的结果

### 关键思路
```
预填充:  [t0 t1 t2 t3] → 完整因果注意力 → cache = (K_{0:3}, V_{0:3})
解码:   [t4]           → Q=t4, K/V=cache+t4  → cache = (K_{0:4}, V_{0:4})
解码:   [t5]           → Q=t5, K/V=cache+t5  → cache = (K_{0:5}, V_{0:5})
```

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import math

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

class KVCacheAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        pass  # Initialize W_q, W_k, W_v, W_o

    def forward(self, x, cache=None):
        # 1. Project Q, K, V from x
        # 2. Reshape to multi-head: (B, num_heads, S, d_k)
        # 3. If cache exists, concat new K/V with cached K/V
        # 4. Compute attention (causal mask needed during prefill)
        # 5. Return (output, (K_all, V_all))
        pass

In [ ]:
# 🧪 Debug
torch.manual_seed(0)
attn = KVCacheAttention(d_model=64, num_heads=4)
x = torch.randn(1, 6, 64)

# Full forward
full_out, _ = attn(x)
print("Full output shape:", full_out.shape)  # (1, 6, 64)

# Incremental: prefill 4, decode 1, decode 1
out1, cache = attn(x[:, :4])
print("Cache K shape:", cache[0].shape)  # (1, 4, 4, 16)
out2, cache = attn(x[:, 4:5], cache=cache)
out3, cache = attn(x[:, 5:6], cache=cache)
inc_out = torch.cat([out1, out2, out3], dim=1)
print("Match:", torch.allclose(full_out, inc_out, atol=1e-5))

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('kv_cache')